# Neo Flood Fill — Algorithm Research & Benchmark Notebook

A reproducible research companion to the **Neo Flood Fill Interactive Algorithm Studio**.
All algorithms, grid models, terrain generators, and benchmarks are imported directly from the modular `src/neo_flood_fill` package — ensuring zero code duplication between research experiments, automated tests (`pytest`), the CLI simulation, and the Pygame / WebAssembly Studio (`main.py`).

### Contents
1. **Setup & Modular Imports**
2. **Two-Layer Grid Model & Terrain Generators** (`Multi-Room`, `Maze`, `Obstacles`, `Spiral`)
3. **Flood Fill Algorithms** (`Recursive DFS`, `Iterative DFS`, `Queue BFS`, `Scanline` — 4-way & 8-way)
4. **Pathfinding Algorithms** (`A*`, `BFS`, `DFS` — Manhattan & Octile heuristics)
5. **Console Step-by-Step Simulation** (non-blocking demo)
6. **Scientific Complexity Benchmarks & Visualizations** ($O(N)$ Time, $O(N)$ vs $O(\sqrt{N})$ Space, CPython `tracemalloc` note)
7. **Launching the Interactive Studio & Running Tests**


## 1. Setup & Modular Imports


In [1]:
import copy
import sys
from pathlib import Path

import matplotlib.pyplot as plt

# Ensure src/ is on sys.path whether or not the package is installed editable
ROOT = Path.cwd()
SRC = ROOT / "src"
if SRC.is_dir() and str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from neo_flood_fill import (
    EMPTY,
    FLOOD_FILL_ALGORITHMS,
    PALETTE_START_ID,
    PATHFINDING_ALGORITHMS,
    WALL,
    Grid,
    create_grid,
    flood_fill_bfs,
    flood_fill_iterative_dfs,
    flood_fill_recursive,
    flood_fill_scanline,
    format_grid,
    generate_maze,
    generate_multi_room,
    generate_random_obstacles,
    generate_spiral,
    pathfind_astar,
    pathfind_bfs,
    pathfind_dfs,
)
from neo_flood_fill.benchmarks import (
    benchmark_flood_fill,
    benchmark_pathfinding,
    compute_scaling_ratios,
)
from neo_flood_fill.console_demo import (
    build_sample_console_grid,
    run_console_flood_fill,
    run_console_pathfinding,
)

plt.rcParams.update({
    "figure.facecolor": "#0d111a",
    "axes.facecolor": "#161b27",
    "axes.edgecolor": "#2e384d",
    "axes.labelcolor": "#e2e8f0",
    "xtick.color": "#94a3b8",
    "ytick.color": "#94a3b8",
    "text.color": "#f1f5f9",
    "grid.color": "#263044",
    "grid.linestyle": "--",
    "grid.alpha": 0.6,
    "legend.facecolor": "#1d2434",
    "legend.edgecolor": "#2e384d",
})

print("neo_flood_fill loaded cleanly from:", SRC)


neo_flood_fill loaded cleanly from: /Users/quocanhngonguyen/Downloads/Neo-Flood-Filling-Game-/src


## 2. Two-Layer Grid Model & Terrain Generators

Unlike v1 (which stored walls, regions, and search states in a single flat integer grid), v2 separates:
- **`base` layer:** `EMPTY`, `WALL`, and multi-color region IDs (`10..21`) so multiple enclosed rooms can hold independent Paint Bucket colors.
- **`overlay` layer:** transient search markers (`START`, `END`, `FRONTIER`, `VISITED`, `PATH`) that can be cleared without erasing painted regions.


In [2]:
terrains = [
    ("Multi-Room (Paint Bucket)", generate_multi_room(17, 23, seed=42)),
    ("Maze (Recursive Backtracking)", generate_maze(17, 23, seed=42)),
    ("Random Obstacles (28%)", generate_random_obstacles(17, 23, density=0.28, seed=42)),
    ("Spiral Corridor (Stress Test)", generate_spiral(17, 17)),
]

fig, axes = plt.subplots(1, 4, figsize=(15, 3.6), dpi=130)
for ax, (title, (matrix, start, end)) in zip(axes, terrains):
    ax.imshow(matrix, cmap="slate" if False else "bone_r", interpolation="nearest")
    ax.scatter([start[1]], [start[0]], c="#22c55e", s=55, label="Start", zorder=3)
    ax.scatter([end[1]], [end[0]], c="#ef4444", s=55, label="End", zorder=3)
    ax.set_title(title, fontsize=10, fontweight="bold")
    ax.set_xticks([])
    ax.set_yticks([])
axes[0].legend(loc="lower right", fontsize=8)
fig.tight_layout()
plt.show()


/var/folders/jt/8mtqg8ps3fqbxsts88d3zrw40000gn/T/ipykernel_1224/1096848912.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Flood Fill Algorithms — Equivalence, Bug Fixes & 4-Way vs 8-Way

We implement four Flood Fill algorithms with identical inputs and unified `AlgorithmMetrics`:
- **Recursive DFS:** intuitive depth-first reference; now guarded against `target_color == fill_color` and catches `RecursionError` safely.
- **Iterative DFS:** explicit stack; immune to Python's recursion ceiling.
- **Queue BFS:** FIFO `deque`; expands layer-by-layer with $O(\sqrt{N})$ peak frontier on open square grids.
- **Scanline Flood Fill:** fills entire horizontal spans per pop and seeds only one coordinate per adjacent run — drastically reducing frontier size.


In [3]:
# 1. Cross-algorithm equivalence check on a random obstacle grid (4-way and 8-way)
template, start, _ = generate_random_obstacles(15, 19, density=0.32, seed=7)
sr, sc = start

for conn in (4, 8):
    print(f"=== Flood Fill Equivalence ({conn}-Way) ===")
    outputs = {}
    for key, (name, fn, _) in FLOOD_FILL_ALGORITHMS.items():
        g = copy.deepcopy(template)
        m = fn(g, sr, sc, EMPTY, PALETTE_START_ID, conn)
        outputs[key] = g
        space = m.peak_recursion_depth if key == "recursive_dfs" else m.peak_frontier
        print(f"  {name:<15} | filled={m.cells_filled:<4} | peak_space={space:<4} | {m.elapsed_ms:.3f} ms")
    ref = outputs["bfs"]
    assert all(g == ref for g in outputs.values()), "All algorithms must match identically!"
    print("  -> All 4 algorithms produced byte-for-byte identical grids!\n")

# 2. Regression check: target_color == fill_color (v1 bug fixed across all 4 variants)
g_guard = create_grid(5, 5, EMPTY)
for key, (name, fn, _) in FLOOD_FILL_ALGORITHMS.items():
    m = fn(g_guard, 2, 2, EMPTY, EMPTY, 4)
    assert m.cells_filled == 0 and not m.recursion_limit_hit
print("Guard verified: target_color == fill_color safely returns 0 changes for all 4 algorithms.")


=== Flood Fill Equivalence (4-Way) ===
  Queue BFS       | filled=135  | peak_space=9    | 0.093 ms
  Scanline        | filled=135  | peak_space=17   | 0.039 ms
  Iterative DFS   | filled=135  | peak_space=43   | 0.099 ms
  Recursive DFS   | filled=135  | peak_space=63   | 0.089 ms
  -> All 4 algorithms produced byte-for-byte identical grids!

=== Flood Fill Equivalence (8-Way) ===
  Queue BFS       | filled=189  | peak_space=21   | 0.211 ms
  Scanline        | filled=189  | peak_space=30   | 0.050 ms
  Iterative DFS   | filled=189  | peak_space=243  | 0.239 ms
  Recursive DFS   | filled=189  | peak_space=124  | 0.222 ms
  -> All 4 algorithms produced byte-for-byte identical grids!

Guard verified: target_color == fill_color safely returns 0 changes for all 4 algorithms.


In [4]:
# 3. Demonstrating 4-Way vs 8-Way connectivity across a diagonal bridge
diag_template = [
    [EMPTY, WALL,  WALL],
    [WALL,  EMPTY, WALL],
    [WALL,  WALL,  EMPTY],
]

g4 = copy.deepcopy(diag_template)
g8 = copy.deepcopy(diag_template)
m4 = flood_fill_scanline(g4, 0, 0, EMPTY, 9, connectivity=4)
m8 = flood_fill_scanline(g8, 0, 0, EMPTY, 9, connectivity=8)

print(f"4-Way Scanline (filled={m4.cells_filled}):")
print(format_grid(g4))
print(f"\n8-Way Scanline (filled={m8.cells_filled}):")
print(format_grid(g8))


4-Way Scanline (filled=1):
9 1 1
1 0 1
1 1 0

8-Way Scanline (filled=3):
9 1 1
1 9 1
1 1 9


## 4. Pathfinding Algorithms — A* vs BFS vs DFS

Comparing three graph-search strategies on the same `21 x 31` Maze and Open Grid:
- **BFS:** guarantees shortest unweighted path (minimum hops).
- **A\* Search:** uses **Manhattan** heuristic (4-way) and **Octile** heuristic with diagonal cost $\sqrt{2}$ (8-way) to find optimal paths while exploring significantly fewer cells.
- **DFS:** serves as a visual contrast baseline — finds a valid route quickly in some corridors, but does **not** guarantee optimality.


In [5]:
maze, m_start, m_end = generate_maze(21, 31, seed=42)
open_g = create_grid(21, 31, EMPTY)

for label, grid_data, s, e in [
    ("Open Grid (21x31, 4-Way)", open_g, (1, 1), (19, 29)),
    ("Maze (21x31, 4-Way)", maze, m_start, m_end),
]:
    print(f"=== {label} ===")
    for key, (name, fn, _) in PATHFINDING_ALGORITHMS.items():
        path, metrics = fn(grid_data, s, e, 4, WALL)
        print(
            f"  {name:<10} | found={str(metrics.found):<5} | visited={metrics.cells_visited:<4} "
            f"| peak={metrics.peak_frontier:<4} | path_len={metrics.path_length:<4} "
            f"| cost={metrics.path_cost:<6.1f} | {metrics.elapsed_ms:.3f} ms"
        )
    print()

# 8-Way Octile A* comparison on Open Grid
_, m_a4 = pathfind_astar(open_g, (1, 1), (19, 29), connectivity=4)
_, m_a8 = pathfind_astar(open_g, (1, 1), (19, 29), connectivity=8)
print(
    f"A* 4-Way (Manhattan): {m_a4.path_length} cells, cost={m_a4.path_cost:.2f} | "
    f"A* 8-Way (Octile): {m_a8.path_length} cells, cost={m_a8.path_cost:.2f}"
)


=== Open Grid (21x31, 4-Way) ===
  A* Search  | found=True  | visited=47   | peak=93   | path_len=47   | cost=46.0   | 0.176 ms
  BFS        | found=True  | visited=647  | peak=23   | path_len=47   | cost=46.0   | 0.942 ms
  DFS        | found=True  | visited=629  | peak=600  | path_len=611  | cost=610.0  | 1.368 ms

=== Maze (21x31, 4-Way) ===
  A* Search  | found=True  | visited=103  | peak=2    | path_len=103  | cost=102.0  | 0.181 ms
  BFS        | found=True  | visited=115  | peak=2    | path_len=103  | cost=102.0  | 0.170 ms
  DFS        | found=True  | visited=131  | peak=4    | path_len=103  | cost=102.0  | 0.197 ms

A* 4-Way (Manhattan): 47 cells, cost=46.00 | A* 8-Way (Octile): 29 cells, cost=35.46


## 5. Console Step-by-Step Simulation (Phase 2 Deliverable)

Running a non-blocking console preview (`delay=0`, capped preview frames so **Run All** completes smoothly). For full interactive coordinate input in your terminal, run:
```bash
python main.py --console --interactive
```


In [6]:
sample_grid = build_sample_console_grid()
run_console_pathfinding(
    sample_grid,
    start=(0, 0),
    end=(5, 6),
    algorithm="astar",
    connectivity=4,
    delay=0.0,
    max_frames=3,
)



=== Console Pathfinding: A* Search (4-way) ===
Start: (0, 0) -> End: (5, 6)

Explore step 1: (1, 0)
S # . . . . .
v # # . . . .
. . . . # . .
. . . . # . .
. . # . . . .
. . . . . . E

Explore step 2: (2, 0)
S # . . . . .
v # # . . . .
v . . . # . .
. . . . # . .
. . # . . . .
. . . . . . E

Explore step 3: (3, 0)
S # . . . . .
v # # . . . .
v . . . # . .
v . . . # . .
. . # . . . .
. . . . . . E

Final grid:
S # . . . . .
* # # . . . .
* . . . # . .
* . . . # . .
* . # . . . .
* * * * * * E

Path found! Length: 12 cells | Cost: 11.00 | Visited: 12 | Time: 0.070 ms


AlgorithmMetrics(algorithm='A*', connectivity=4, cells_visited=12, cells_filled=0, peak_frontier=7, peak_recursion_depth=0, path_length=12, path_cost=11.0, elapsed_ms=0.07012509740889072, found=True, recursion_limit_hit=False)

## 6. Scientific Complexity Benchmarks & Visualizations

### Key Fix Over v1: Accurate Space Complexity Measurement
In v1, using `tracemalloc` alone caused Recursive DFS to report `0.01 KB` vs BFS `1.34 KB`. That happened because **`tracemalloc` only tracks Python heap allocations** (such as `deque` blocks and coordinate tuples), whereas CPython 3.11+ allocates call-stack frames on a contiguous interpreter frame stack.

In v2, `neo_flood_fill.benchmarks` separates:
1. **Algorithmic Space:** `peak_recursion_depth` ($O(N)$ for Recursive DFS) vs `peak_frontier` ($O(\sqrt{N})$ on open grids for BFS, and near-constant row seeds for Scanline).
2. **Heap Peak (`tracemalloc`):** measured on an isolated pass so tracing overhead never distorts wall-clock timings.


In [7]:
sizes = (15, 25, 35, 50, 65)
open_ff = benchmark_flood_fill(
    sizes=sizes,
    terrains=("open",),
    algorithms=("recursive_dfs", "iterative_dfs", "bfs", "scanline"),
    repeats=5,
)

print(f"{'Algorithm':<15} {'Grid':<8} {'N':>6} {'Time (ms)':>10} {'Peak Space':>11} {'Heap (KB)':>10}")
print("-" * 66)
for r in open_ff:
    peak_space = r.peak_recursion_depth if r.algorithm_key == "recursive_dfs" else r.peak_frontier
    print(
        f"{r.algorithm:<15} {r.rows}x{r.cols:<5} {r.total_cells:>6} "
        f"{r.median_ms:>10.3f} {peak_space:>11} {r.heap_peak_kb:>10.2f}"
    )

print("\n--- Linear O(N) Time Scaling Verification (Open Grid) ---")
for key in ("bfs", "scanline", "iterative_dfs"):
    subset = [r for r in open_ff if r.algorithm_key == key]
    ratios = compute_scaling_ratios(subset)
    formatted = ", ".join(f"{lbl}: {t_r:.2f}x (N {c_r:.2f}x)" for lbl, c_r, t_r in ratios)
    print(f"  {key:<14} -> {formatted}")


Algorithm       Grid          N  Time (ms)  Peak Space  Heap (KB)
------------------------------------------------------------------
Recursive DFS   15x15       225      0.142         225      11.19
Iterative DFS   15x15       225      0.163         197       1.77
Queue BFS       15x15       225      0.125          15       1.47
Scanline        15x15       225      0.025           1       0.25
Recursive DFS   25x25       625      0.477         625      41.48
Iterative DFS   25x25       625      0.475         577       4.93
Queue BFS       25x25       625      0.362          25       1.71
Scanline        25x25       625      0.062           1       0.31
Recursive DFS   35x35      1225      0.837        1225      88.94
Iterative DFS   35x35      1225      0.932        1157      10.02
Queue BFS       35x35      1225      0.709          35       1.56
Scanline        35x35      1225      0.125           1       0.31
Recursive DFS   50x50      2500      1.837        2500     188.16
Iterative

In [8]:
terrain_ff = benchmark_flood_fill(
    sizes=(35,),
    terrains=("open", "maze", "spiral"),
    algorithms=("recursive_dfs", "iterative_dfs", "bfs", "scanline"),
    repeats=3,
)

algo_styles = {
    "recursive_dfs": ("Recursive DFS", "#f43f5e", "o"),
    "iterative_dfs": ("Iterative DFS", "#f97316", "s"),
    "bfs": ("Queue BFS", "#38bdf8", "^"),
    "scanline": ("Scanline", "#20c997", "D"),
}

fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.5), dpi=130)
fig.suptitle("Flood Fill Benchmarks — Time O(N) & Algorithmic Space", fontsize=13, fontweight="bold")

# Panel 1: Runtime vs N
ax = axes[0]
for key, (label, color, marker) in algo_styles.items():
    sub = [r for r in open_ff if r.algorithm_key == key]
    ax.plot([r.total_cells for r in sub], [r.median_ms for r in sub], label=label, color=color, marker=marker, lw=2)
ax.set_title("Open Grid Runtime vs N")
ax.set_xlabel("Total Cells N")
ax.set_ylabel("Median Runtime (ms)")
ax.grid(True)
ax.legend(fontsize=8)

# Panel 2: Peak Stack / Frontier vs N
ax = axes[1]
for key, (label, color, marker) in algo_styles.items():
    sub = [r for r in open_ff if r.algorithm_key == key]
    ys = [r.peak_recursion_depth if key == "recursive_dfs" else r.peak_frontier for r in sub]
    ax.plot([r.total_cells for r in sub], ys, label=label, color=color, marker=marker, lw=2)
ax.set_title("Peak Stack / Frontier vs N (Log Scale)")
ax.set_xlabel("Total Cells N")
ax.set_ylabel("Peak Entries")
ax.set_yscale("log")
ax.grid(True)
ax.legend(fontsize=8)

# Panel 3: Terrain Comparison (35x35)
ax = axes[2]
terr_names = ("open", "maze", "spiral")
xs = list(range(len(terr_names)))
w = 0.19
for idx, (key, (label, color, _)) in enumerate(algo_styles.items()):
    vals = [next(r.median_ms for r in terrain_ff if r.terrain == t and r.algorithm_key == key) for t in terr_names]
    ax.bar([x + (idx - 1.5) * w for x in xs], vals, width=w, label=label, color=color)
ax.set_title("Runtime Across Terrains (35x35)")
ax.set_xticks(xs)
ax.set_xticklabels(["Open Grid", "Maze", "Spiral"])
ax.set_ylabel("Median Runtime (ms)")
ax.grid(True, axis="y")
ax.legend(fontsize=8)

fig.tight_layout(rect=(0, 0, 1, 0.93))
plt.show()


/var/folders/jt/8mtqg8ps3fqbxsts88d3zrw40000gn/T/ipykernel_1224/1638902938.py:58: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
pf_bench = benchmark_pathfinding(
    sizes=(31,),
    terrains=("open", "maze", "spiral"),
    algorithms=("astar", "bfs", "dfs"),
    connectivity=4,
    repeats=5,
)

pf_styles = {
    "astar": ("A* (Manhattan)", "#20c997"),
    "bfs": ("BFS", "#38bdf8"),
    "dfs": ("DFS", "#f43f5e"),
}

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2), dpi=130)
fig.suptitle("Pathfinding Benchmarks (31x31, 4-Way) — A* vs BFS vs DFS", fontsize=13, fontweight="bold")
xs = [0, 1, 2]
w = 0.24

for panel_idx, (metric_attr, title, ylabel) in enumerate([
    ("cells_visited", "Cells Explored (Lower = Better)", "Cells Visited"),
    ("path_length", "Path Length (A* & BFS = Optimal)", "Cells in Path"),
]):
    ax = axes[panel_idx]
    for idx, (key, (label, color)) in enumerate(pf_styles.items()):
        vals = [getattr(next(r for r in pf_bench if r.terrain == t and r.algorithm_key == key), metric_attr) for t in ("open", "maze", "spiral")]
        ax.bar([x + (idx - 1) * w for x in xs], vals, width=w, label=label, color=color)
    ax.set_title(title)
    ax.set_xticks(xs)
    ax.set_xticklabels(["Open Grid", "Maze", "Spiral"])
    ax.set_ylabel(ylabel)
    ax.grid(True, axis="y")
    ax.legend(fontsize=9)

fig.tight_layout(rect=(0, 0, 1, 0.93))
plt.show()


/var/folders/jt/8mtqg8ps3fqbxsts88d3zrw40000gn/T/ipykernel_1224/214218683.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Interactive Pygame Studio, WebAssembly Demo & Test Suite

Because Pygame opens a native OS event loop (which can conflict with Jupyter kernels on macOS/Windows), the Interactive Algorithm Studio runs as a standalone async application outside the notebook:

- **Launch Desktop Studio:**
  ```bash
  python main.py
  ```
- **Run Console CLI Demo:**
  ```bash
  python main.py --console --mode path --algo astar
  python main.py --console --interactive
  ```
- **Run Automated Test Suite (79 tests):**
  ```bash
  pytest -v
  ```
- **Build WebAssembly Bundle (`pygbag`):**
  ```bash
  python scripts/build_web.py
  ```
- **Live Browser Demo (GitHub Pages):**
  [https://alejandronguyen1504.github.io/Neo-Flood-Filling-Game-/](https://alejandronguyen1504.github.io/Neo-Flood-Filling-Game-/)
